<a href="https://colab.research.google.com/github/GitHubAman2004/stock_price_predictor/blob/solved-6-problems-in-extended-dataset/latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q yfinance pandas scikit-learn torch

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ---- Configuration ------------------------------------------------
TICKER      = "AAPL"
START_DATE  = "2005-01-01"   # see note below
SEQ_LEN     = 60             # days of history the model sees
THRESHOLD   = 0.002          # 0.2% - the bar for "it went up"
TRAIN_FRAC  = 0.80
BATCH_SIZE  = 32
EPOCHS      = 15
LR          = 1e-3
# -------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
raw = yf.download(TICKER, start=START_DATE, auto_adjust=True, progress=False)

# yfinance returns a MultiIndex column header when given a ticker. Flatten it.
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = pd.DataFrame(index=raw.index)

# --- Stationary features (all roughly scale-free) ---
df["Ret_1"]       = raw["Close"].pct_change()                                  # yesterday's return
df["Ret_5"]       = raw["Close"].pct_change(5)                                 # 1-week return
df["SMA10_ratio"] = raw["Close"] / raw["Close"].rolling(10).mean() - 1         # distance from 10d average
df["SMA50_ratio"] = raw["Close"] / raw["Close"].rolling(50).mean() - 1         # distance from 50d average
df["Volatility"]  = df["Ret_1"].rolling(10).std()                              # recent choppiness
df["HL_range"]    = (raw["High"] - raw["Low"]) / raw["Close"]                   # intraday range
df["CO_ratio"]    = (raw["Close"] - raw["Open"]) / raw["Open"]                  # intraday drift
df["Vol_z"]       = raw["Volume"] / raw["Volume"].rolling(20).mean() - 1       # unusual volume

# --- Kept for LABELS ONLY. Never fed to the model. ---
df["RawClose"] = raw["Close"]

df = df.dropna()

FEATURES = ["Ret_1", "Ret_5", "SMA10_ratio", "SMA50_ratio",
            "Volatility", "HL_range", "CO_ratio", "Vol_z"]
N_FEATURES = len(FEATURES)

print(f"Rows: {len(df)}   Features: {N_FEATURES}")
print(f"Date range: {df.index[0].date()} to {df.index[-1].date()}\n")
print(df[FEATURES].describe().T[["mean", "std", "min", "max"]].round(4))

Rows: 5362   Features: 8
Date range: 2005-03-15 to 2026-07-08

               mean     std     min     max
Ret_1        0.0012  0.0201 -0.1792  0.1533
Ret_5        0.0062  0.0439 -0.2431  0.1983
SMA10_ratio  0.0049  0.0331 -0.1905  0.1339
SMA50_ratio  0.0270  0.0818 -0.4078  0.2810
Volatility   0.0177  0.0097  0.0033  0.0807
HL_range     0.0229  0.0149  0.0042  0.2396
CO_ratio     0.0003  0.0164 -0.1200  0.1564
Vol_z        0.0039  0.3799 -0.7644  4.2456


In [ ]:
split_row = int(len(df) * TRAIN_FRAC)

scaler = StandardScaler()
scaler.fit(df[FEATURES].iloc[:split_row])      # <-- training rows only
scaled = scaler.transform(df[FEATURES])        # <-- applied to all rows

raw_close = df["RawClose"].values               # untouched real prices, for labels

print(f"Scaler fit on rows 0 to {split_row-1}  ({df.index[split_row-1].date()})")
print(f"Test period begins at row {split_row} ({df.index[split_row].date()})")

Scaler fit on rows 0 to 4288  (2022-03-25)
Test period begins at row 4289 (2022-03-28)


In [ ]:
X_list, y_list, split_flag = [], [], []
purged = 0                                                    # <-- NEW: a counter

for i in range(SEQ_LEN - 1, len(df) - 1):
    window = scaled[i - SEQ_LEN + 1 : i + 1]                   # days i-59 ... i  (Fix 4)

    future_ret = (raw_close[i + 1] - raw_close[i]) / raw_close[i]  # RAW prices   (Fix 2)
    label = 1 if future_ret > THRESHOLD else 0

    if i + 1 < split_row:                       # label day is inside training era
        flag = "train"
    elif i - SEQ_LEN + 1 >= split_row:          # whole window is inside test era
        flag = "test"
    else:
        purged += 1                                            # <-- NEW: count it
        continue                                # straddles the boundary -> discard

    X_list.append(window)
    y_list.append(label)
    split_flag.append(flag)

X = torch.tensor(np.array(X_list), dtype=torch.float32)
y = torch.tensor(np.array(y_list), dtype=torch.float32).unsqueeze(1)
flags = np.array(split_flag)

X_train, y_train = X[flags == "train"], y[flags == "train"]
X_test,  y_test  = X[flags == "test"],  y[flags == "test"]

print(f"X shape: {tuple(X.shape)}   (samples, days, features)")
print(f"Train: {len(X_train)}   Test: {len(X_test)}   Purged: {purged}")   # <-- NEW
print(f"Positive rate  train: {y_train.mean():.4f}   test: {y_test.mean():.4f}")

X shape: (5242, 60, 8)   (samples, days, features)
Train: 4229   Test: 1013   Purged: 60
Positive rate  train: 0.4729   test: 0.4748


In [ ]:
pos_rate_test = y_test.mean().item()
majority_baseline = max(pos_rate_test, 1 - pos_rate_test)
always_up = pos_rate_test

print(f"Test positive rate      : {pos_rate_test*100:.2f}%")
print(f"Always predict 'down/flat': {(1-pos_rate_test)*100:.2f}%")
print(f"Always predict 'up'       : {always_up*100:.2f}%")
print(f"\n>>> MAJORITY BASELINE TO BEAT: {majority_baseline*100:.2f}% <<<")
print("\nIf the trained model scores below this, it has learned nothing.")

Test positive rate      : 47.48%
Always predict 'down/flat': 52.52%
Always predict 'up'       : 47.48%

>>> MAJORITY BASELINE TO BEAT: 52.52% <<<

If the trained model scores below this, it has learned nothing.


In [ ]:
train_ds = TensorDataset(X_train, y_train)
test_ds  = TensorDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"{len(train_loader)} training batches, {len(test_loader)} test batches")

133 training batches, 32 test batches


In [ ]:
class HybridForecastingModel(nn.Module):
    def __init__(self, n_features=N_FEATURES, cnn_ch=16, hidden=32, heads=4, dropout=0.2):
        super().__init__()
        self.cnn       = nn.Conv1d(in_channels=n_features, out_channels=cnn_ch,
                                   kernel_size=3, padding=1)
        self.relu      = nn.ReLU()
        self.dropout   = nn.Dropout(dropout)
        self.lstm      = nn.LSTM(input_size=cnn_ch, hidden_size=hidden, batch_first=True)
        self.attention = nn.MultiheadAttention(embed_dim=hidden, num_heads=heads,
                                               batch_first=True)
        self.fc        = nn.Linear(hidden, 1)

    def forward(self, x):                 # x: (batch, days, features)
        x = x.permute(0, 2, 1)            #    (batch, features, days)
        x = self.relu(self.cnn(x))        #    (batch, cnn_ch, days)
        x = self.dropout(x)
        x = x.permute(0, 2, 1)            #    (batch, days, cnn_ch)

        x, _ = self.lstm(x)               #    (batch, days, hidden)
        x, _ = self.attention(x, x, x)    #    (batch, days, hidden)

        last_day = x[:, -1, :]            #    (batch, hidden)
        return self.fc(last_day)          #    (batch, 1)  <- logits


model = HybridForecastingModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTrainable parameters: {n_params:,}")

HybridForecastingModel(
  (cnn): Conv1d(8, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (lstm): LSTM(16, 32, batch_first=True)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
  )
  (fc): Linear(in_features=32, out_features=1, bias=True)
)

Trainable parameters: 11,057


In [ ]:
pos = (y_train == 1).sum().item()
neg = (y_train == 0).sum().item()
pos_weight = torch.tensor([neg / pos], device=device)

print(f"Train positives: {pos}   negatives: {neg}   pos_weight: {neg/pos:.3f}\n")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

# The loss a no-information (constant-output) model settles at, GIVEN pos_weight.
r = pos / (pos + neg)
NO_INFO_LOSS = 2 * (1 - r) * np.log(2)
print(f"No-information loss floor : {NO_INFO_LOSS:.4f}")
print(f"Unweighted coin flip     : {np.log(2):.4f}")
print()

for epoch in range(EPOCHS):
    model.train()                                   # dropout ON
    running_loss, correct, total = 0.0, 0, 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += ((logits > 0).float() == batch_y).sum().item()
        total   += batch_y.numel()

    epoch_loss = running_loss / len(train_loader)
    marker = "  <-- learning" if epoch_loss < NO_INFO_LOSS - 1e-3 else "  (no signal yet)"
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | train loss {epoch_loss:.4f} | "
          f"train acc {correct/total*100:5.2f}%{marker}")

print()
print(f"Floor is {NO_INFO_LOSS:.4f}. A loss stuck there means the model gave up.")

Train positives: 2000   negatives: 2229   pos_weight: 1.115

No-information loss floor : 0.7307
Unweighted coin flip     : 0.6931

Epoch  1/15 | train loss 0.7311 | train acc 49.75%  (no signal yet)
Epoch  2/15 | train loss 0.7311 | train acc 48.85%  (no signal yet)
Epoch  3/15 | train loss 0.7311 | train acc 48.92%  (no signal yet)
Epoch  4/15 | train loss 0.7310 | train acc 49.21%  (no signal yet)
Epoch  5/15 | train loss 0.7309 | train acc 49.37%  (no signal yet)
Epoch  6/15 | train loss 0.7309 | train acc 49.54%  (no signal yet)
Epoch  7/15 | train loss 0.7309 | train acc 49.40%  (no signal yet)
Epoch  8/15 | train loss 0.7308 | train acc 49.56%  (no signal yet)
Epoch  9/15 | train loss 0.7307 | train acc 49.82%  (no signal yet)
Epoch 10/15 | train loss 0.7307 | train acc 49.80%  (no signal yet)
Epoch 11/15 | train loss 0.7306 | train acc 50.27%  (no signal yet)
Epoch 12/15 | train loss 0.7306 | train acc 50.34%  (no signal yet)
Epoch 13/15 | train loss 0.7305 | train acc 50.63%  (

In [ ]:
model.eval()                                     # dropout OFF
tp = fp = tn = fn = 0
test_loss = 0.0

with torch.no_grad():                            # no gradient graph
    for batch_X, batch_y in test_loader:         # <-- the TEST loader, at last
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        logits = model(batch_X)
        test_loss += criterion(logits, batch_y).item()
        pred = (logits > 0).float()

        tp += ((pred == 1) & (batch_y == 1)).sum().item()
        fp += ((pred == 1) & (batch_y == 0)).sum().item()
        fn += ((pred == 0) & (batch_y == 1)).sum().item()
        tn += ((pred == 0) & (batch_y == 0)).sum().item()

n = tp + tn + fp + fn
eps = 1e-9
accuracy  = (tp + tn) / n
precision = tp / (tp + fp + eps)
recall    = tp / (tp + fn + eps)
f1        = 2 * precision * recall / (precision + recall + eps)

print("=" * 52)
print("           TEST SET RESULTS (unseen data)")
print("=" * 52)
print(f"Test loss   : {test_loss/len(test_loader):.4f}   (no-info floor = {NO_INFO_LOSS:.4f})")
print(f"Accuracy    : {accuracy*100:6.2f}%")
print(f"Precision   : {precision:6.3f}")
print(f"Recall      : {recall:6.3f}")
print(f"F1          : {f1:6.3f}")
print("-" * 52)
print("Confusion matrix")
print(f"                 predicted up   predicted down")
print(f"  actual up      {tp:>10}   {fn:>14}")
print(f"  actual down    {fp:>10}   {tn:>14}")
print("-" * 52)
print(f"Majority baseline : {majority_baseline*100:6.2f}%")
print(f"Model accuracy    : {accuracy*100:6.2f}%")
edge = (accuracy - majority_baseline) * 100
verdict = "BEATS baseline" if edge > 0 else "LOSES to baseline"
print(f"Edge              : {edge:+6.2f} pp   -> {verdict}")
print("=" * 52)

           TEST SET RESULTS (unseen data)
Test loss   : 0.7304   (no-info floor = 0.7307)
Accuracy    :  52.32%
Precision   :  0.498
Recall      :  0.659
F1          :  0.568
----------------------------------------------------
Confusion matrix
                 predicted up   predicted down
  actual up             317              164
  actual down           319              213
----------------------------------------------------
Majority baseline :  52.52%
Model accuracy    :  52.32%
Edge              :  -0.20 pp   -> LOSES to baseline


In [ ]:
CKPT = "model.pt"

torch.save({
    "state_dict":  model.state_dict(),   # the learned numbers
    "scaler":      scaler,               # the normalisation rules
    "features":    FEATURES,             # column names, in order
    "seq_len":     SEQ_LEN,
    "threshold":   THRESHOLD,
    "ticker":      TICKER,
    "test_accuracy": accuracy,
    "baseline":      majority_baseline,
}, CKPT)

print(f"Saved -> {CKPT}")

# In Colab, download it to your machine:
# from google.colab import files; files.download(CKPT)

Saved -> model.pt


In [ ]:
def load_model(path=CKPT):
    ckpt = torch.load(path, weights_only=False, map_location=device)
    m = HybridForecastingModel(n_features=len(ckpt["features"])).to(device)
    m.load_state_dict(ckpt["state_dict"])
    m.eval()                                  # always. dropout off.
    return m, ckpt


def build_features(ticker, start="2020-01-01"):
    """Same feature pipeline as training. Must match exactly."""
    r = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if isinstance(r.columns, pd.MultiIndex):
        r.columns = r.columns.get_level_values(0)
    d = pd.DataFrame(index=r.index)
    d["Ret_1"]       = r["Close"].pct_change()
    d["Ret_5"]       = r["Close"].pct_change(5)
    d["SMA10_ratio"] = r["Close"] / r["Close"].rolling(10).mean() - 1
    d["SMA50_ratio"] = r["Close"] / r["Close"].rolling(50).mean() - 1
    d["Volatility"]  = d["Ret_1"].rolling(10).std()
    d["HL_range"]    = (r["High"] - r["Low"]) / r["Close"]
    d["CO_ratio"]    = (r["Close"] - r["Open"]) / r["Open"]
    d["Vol_z"]       = r["Volume"] / r["Volume"].rolling(20).mean() - 1
    return d.dropna(), r


def predict_tomorrow(ticker=TICKER):
    m, ckpt = load_model()
    feats, prices = build_features(ticker)

    window = feats[ckpt["features"]].tail(ckpt["seq_len"])   # (60, 8)
    if len(window) < ckpt["seq_len"]:
        raise ValueError(f"Need {ckpt['seq_len']} days, got {len(window)}")

    scaled_w = ckpt["scaler"].transform(window)                     # transform, NOT fit_transform
    x = torch.tensor(scaled_w, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 60, 8)

    with torch.no_grad():
        logit = m(x)
        prob  = torch.sigmoid(logit).item()      # logit -> probability

    last_date  = feats.index[-1].date()
    last_close = float(prices["Close"].iloc[-1])

    print(f"Ticker            : {ticker}")
    print(f"Last close        : {last_close:.2f}  ({last_date})")
    print(f"P(up > {ckpt['threshold']*100:.1f}% tomorrow) : {prob*100:.2f}%")
    print(f"Decision          : {'UP' if prob > 0.5 else 'NOT UP'}")
    print(f"\n(model test accuracy {ckpt['test_accuracy']*100:.2f}% vs "
          f"baseline {ckpt['baseline']*100:.2f}%)")
    return prob


_ = predict_tomorrow("AAPL")

Ticker            : AAPL
Last close        : 313.39  (2026-07-08)
P(up > 0.2% tomorrow) : 49.62%
Decision          : NOT UP

(model test accuracy 52.32% vs baseline 52.52%)
